# Example 5 — AI Cafeteria Crowd Forecaster
**Predict attendance, evaluate the forecast, and communicate it responsibly**  
**Difficulty:** Intermediate

Imagine that you help manage a busy dining facility near a university campus. Every operating day, the team must estimate how many customers may arrive.

Preparing too little can create long waits and shortages. Preparing too much can create waste and unnecessary cost.

You have historical data describing earlier operating days. Your goal is to build a machine-learning application that forecasts the number of visitors on a future day.

### Your mission

1. Explore the historical attendance data.
2. identify useful model inputs.
3. split earlier and later dates correctly.
4. compare regression against a simple baseline.
5. train and evaluate a scikit-learn model.
6. inspect prediction errors and model weights.
7. enter a future operating-day scenario.


## Colab use

Run the cells from top to bottom. Free Google Colab CPU is sufficient.

The instructor should configure the dataset location once in the data-loading cell. Students should not need to edit that setup.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

print('Libraries imported successfully.')

## Load the dataset

The prepared dataset is stored as a CSV file. Each row represents one day when the dining facility was open.

The instructor can use either:

- a direct public link to the CSV file, or
- a shared file in Google Drive.

Only the configuration values at the top of the next cell should need editing.

In [ ]:
DATA_URL = ''

# Edit the shared Drive path.
DRIVE_DATA_FILE = (
    '/content/drive/MyDrive/Everything USF/[Internal] USF AI Summer School 2026/New Materials/Group Projects -- Updated/data/'
    'restaurant_visitors.csv'
)


if DATA_URL.strip():
    DATA_FILE = Path('/content/cafeteria_visitors.csv')
    urlretrieve(DATA_URL, DATA_FILE)
    print('Using direct dataset download.')

else:
    try:
        from google.colab import drive
    except ModuleNotFoundError as error:
        raise RuntimeError(
            'This Drive-loading option must run in Google Colab. '
            'Alternatively, set DATA_URL to a direct-download link.'
        ) from error

    drive.mount('/content/drive')
    DATA_FILE = Path(DRIVE_DATA_FILE)
    print('Using the dataset stored in Google Drive.')


print('Dataset location:')
print(DATA_FILE)

In [ ]:
REQUIRED_COLUMNS = [
    'date',
    'day_of_week',
    'month',
    'is_weekend',
    'is_holiday',
    'reservation_status',
    'reserved_visitors',
    'reservation_count',
    'visitors',
]


def load_cafeteria_data(csv_path):
    """Load, validate, and chronologically sort the cafeteria data."""
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(
            'The dataset was not found.\n'
            f'Expected location: {csv_path}\n\n'
            'Ask the instructor for the shared Drive path or direct URL.'
        )

    loaded_data = pd.read_csv(
        csv_path,
        parse_dates=['date'],
    )

    missing_columns = sorted(
        set(REQUIRED_COLUMNS)
        - set(loaded_data.columns)
    )

    if missing_columns:
        raise ValueError(
            'The CSV is missing required columns: '
            + ', '.join(missing_columns)
        )

    loaded_data = (
        loaded_data
        .sort_values('date')
        .reset_index(drop=True)
    )

    return loaded_data


data = load_cafeteria_data(DATA_FILE)

print('Dataset loaded successfully.')
print(f'Rows: {data.shape[0]:,}')
print(f'Columns: {data.shape[1]}')
print(
    'Date range:',
    data['date'].min().date(),
    'through',
    data['date'].max().date(),
)

In [ ]:
display(data.head(10))

## Data dictionary

| Column | Meaning |
|---|---|
| `date` | Date of the operating day |
| `day_of_week` | Name of the weekday |
| `month` | Month number from 1 through 12 |
| `is_weekend` | 1 for Saturday or Sunday; otherwise 0 |
| `is_holiday` | 1 for a holiday; otherwise 0 |
| `reservation_status` | Whether advance reservations were recorded |
| `reserved_visitors` | Total visitors included in reservations known before the day began |
| `reservation_count` | Number of advance reservations |
| `visitors` | Actual number of visitors who arrived |

The column `visitors` is the **target** or **label**.

The model's other selected columns are its **features** or **inputs**.

Missing calendar dates should not be treated as days with zero visitors. They may be dates when the facility was closed.

In [ ]:
print('COLUMN TYPES')
print('=' * 60)
display(data.dtypes.to_frame('data_type'))

print('\nMISSING VALUES')
print('=' * 60)
display(data.isna().sum().to_frame('missing_values'))

print('\nDUPLICATE DATES')
print('=' * 60)
print(data['date'].duplicated().sum())

In [ ]:
numeric_columns = [
    'month',
    'is_weekend',
    'is_holiday',
    'reserved_visitors',
    'reservation_count',
    'visitors',
]

display(
    data[numeric_columns]
    .describe()
    .round(2)
)

In [ ]:
average_visitors = data['visitors'].mean()
minimum_visitors = data['visitors'].min()
maximum_visitors = data['visitors'].max()

largest_crowd_index = data['visitors'].idxmax()
largest_crowd_day = data.loc[largest_crowd_index]

print(f'Average visitors: {average_visitors:.2f}')
print(f'Minimum visitors: {minimum_visitors}')
print(f'Maximum visitors: {maximum_visitors}')

print('\nDay with the largest observed crowd:')
display(largest_crowd_day.to_frame('value'))

# Exploratory Data Analysis

Before training a model, investigate the data.

Look for:

- changes in attendance over time,
- differences among weekdays,
- differences between holidays and other operating days,
- relationships between reservations and attendance,
- extreme or unusual observations.

Exploration can suggest useful features, but a visible relationship does not automatically prove causation.

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    data['date'],
    data['visitors'],
    linewidth=1,
)

plt.xlabel('Date')
plt.ylabel('Actual visitors')
plt.title('Daily Attendance Over Time')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    data['visitors'],
    bins=25,
    edgecolor='black',
)

plt.xlabel('Number of visitors')
plt.ylabel('Number of operating days')
plt.title('Distribution of Daily Attendance')
plt.tight_layout()
plt.show()

In [ ]:
WEEKDAY_ORDER = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday',
]

weekday_summary = (
    data
    .groupby('day_of_week')['visitors']
    .agg(['count', 'mean', 'median', 'std'])
    .reindex(WEEKDAY_ORDER)
)

display(weekday_summary.round(2))

plt.figure(figsize=(10, 5))

plt.bar(
    weekday_summary.index,
    weekday_summary['mean'],
)

plt.xlabel('Day of week')
plt.ylabel('Average visitors')
plt.title('Average Attendance by Day of Week')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
holiday_summary = (
    data
    .groupby('is_holiday')['visitors']
    .agg(['count', 'mean', 'median', 'std'])
    .rename(
        index={
            0: 'Not a holiday',
            1: 'Holiday',
        }
    )
)

display(holiday_summary.round(2))

plt.figure(figsize=(7, 5))

plt.bar(
    holiday_summary.index,
    holiday_summary['mean'],
)

plt.ylabel('Average visitors')
plt.title('Attendance on Holidays and Non-Holidays')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    data['reserved_visitors'],
    data['visitors'],
    alpha=0.55,
)

plt.xlabel('Visitors included in advance reservations')
plt.ylabel('Actual visitors')
plt.title('Advance Reservations and Actual Attendance')
plt.tight_layout()
plt.show()

reservation_correlation = data[
    ['reserved_visitors', 'visitors']
].corr().iloc[0, 1]

print(
    'Correlation between reserved visitors '
    f'and actual visitors: {reservation_correlation:.3f}'
)

In [ ]:
reservation_status_summary = (
    data
    .groupby('reservation_status')['visitors']
    .agg(['count', 'mean', 'median', 'std'])
)

display(reservation_status_summary.round(2))

plt.figure(figsize=(9, 5))

plt.bar(
    reservation_status_summary.index,
    reservation_status_summary['mean'],
)

plt.xlabel('Reservation status')
plt.ylabel('Average visitors')
plt.title('Average Attendance by Reservation Status')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Regression in Friendly Language

Regression predicts a numerical value. Here, the target is the number of visitors.

A linear regression model has the general form

$
\hat{y}
=
b
+
w_1x_1
+
w_2x_2
+\cdots+
w_px_p,
$

where:

- $\hat{y}$ is the predicted attendance,
- $b$ is the intercept,
- each $x_j$ is a model input,
- each $w_j$ is a learned weight.

### One-hot encoding

A weekday such as `Monday` is text. A regression model needs numbers. One-hot encoding creates columns such as:

- `day_of_week_Monday`
- `day_of_week_Tuesday`
- `day_of_week_Wednesday`

A row receives 1 for the matching category and 0 for the others.

### Mean absolute error

$
\operatorname{MAE}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left|y_i-\hat{y}_i\right|.
$

MAE is the model's average absolute mistake, measured in visitors.

### Root mean squared error

$
\operatorname{RMSE}
=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
(y_i-\hat{y}_i)^2
}.
$

RMSE penalizes large mistakes more strongly than MAE.

### R-squared

$
R^2
=
1-
\frac{
\sum_i(y_i-\hat{y}_i)^2
}{
\sum_i(y_i-\bar{y})^2
}.
$

Larger values usually indicate that the model explains more variation in the held-out data. It does not prove that the model is correct or causal.

# Building the Crowd-Forecasting Model

We will use:

- `day_of_week`
- `month`
- `is_holiday`
- `reserved_visitors`
- `reservation_count`

We will not include every available column.

- `is_weekend` is already determined by `day_of_week`.
- `reservation_status` is already determined by whether advance reservations exist.

Including redundant information can make interpretation harder without adding useful new evidence.

## Why split chronologically?

The rows represent dates. To simulate a real forecasting task, train on earlier dates and test on later dates.

A random split could let later observations influence a model evaluated on earlier observations.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from sklearn.base import clone

print('Machine-learning tools imported.')

In [ ]:
CATEGORICAL_FEATURES = [
    'day_of_week',
    'month',
]

NUMERIC_FEATURES = [
    'is_holiday',
    'reserved_visitors',
    'reservation_count',
]

FEATURES = (
    CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
)

TARGET = 'visitors'
TRAIN_FRACTION = 0.80

print('Model features:')
print(FEATURES)

print('\nPrediction target:')
print(TARGET)

## STUDENT TODO 1 — Make a chronological train/test split

Use the first 80% of the rows for training and the final 20% for testing.

Create:

- `split_index`
- `train_data`
- `test_data`
- `X_train`
- `y_train`
- `X_test`
- `y_test`

Do not shuffle the rows.

In [ ]:
# STUDENT TODO 1: CREATE THE CHRONOLOGICAL SPLIT
# Hints:
#   split_index = int(len(data) * TRAIN_FRACTION)
#   train_data = data.iloc[:split_index].copy()
#   test_data = data.iloc[split_index:].copy()
#   X_train and X_test should contain FEATURES.
#   y_train and y_test should contain TARGET.

split_index = None
train_data = None
test_data = None

X_train = None
y_train = None
X_test = None
y_test = None

print('Create the chronological training and test sets.')

In [ ]:
def check_chronological_split(
    split_index,
    train_data,
    test_data,
    X_train,
    y_train,
    X_test,
    y_test,
):
    values = [
        split_index,
        train_data,
        test_data,
        X_train,
        y_train,
        X_test,
        y_test,
    ]

    if any(value is None for value in values):
        print('The chronological split is not complete.')
        return False

    expected_split = int(
        len(data) * TRAIN_FRACTION
    )

    correct_sizes = (
        split_index == expected_split
        and len(train_data) == expected_split
        and len(test_data) == len(data) - expected_split
    )

    correct_columns = (
        list(X_train.columns) == FEATURES
        and list(X_test.columns) == FEATURES
        and y_train.name == TARGET
        and y_test.name == TARGET
    )

    chronological = (
        train_data['date'].max()
        < test_data['date'].min()
    )

    print('Training rows:', len(train_data))
    print('Testing rows:', len(test_data))

    print(
        'Training date range:',
        train_data['date'].min().date(),
        'through',
        train_data['date'].max().date(),
    )

    print(
        'Testing date range:',
        test_data['date'].min().date(),
        'through',
        test_data['date'].max().date(),
    )

    passed = (
        correct_sizes
        and correct_columns
        and chronological
    )

    print(
        'Passed!'
        if passed
        else 'Check the row order, sizes, and selected columns.'
    )

    return passed


split_ready = check_chronological_split(
    split_index,
    train_data,
    test_data,
    X_train,
    y_train,
    X_test,
    y_test,
)

## Establish a baseline

A model is useful only when it improves on a simple alternative.

Our baseline predicts the **training-set average attendance for every test day**.

In [ ]:
baseline_value = None
baseline_predictions = None
BASELINE_MAE = None

if split_ready:
    baseline_value = y_train.mean()

    baseline_predictions = np.full(
        shape=len(y_test),
        fill_value=baseline_value,
    )

    BASELINE_MAE = mean_absolute_error(
        y_test,
        baseline_predictions,
    )

    print(
        'Baseline prediction for every test day:',
        f'{baseline_value:.2f} visitors',
    )

    print(
        'Baseline mean absolute error:',
        f'{BASELINE_MAE:.2f} visitors',
    )

else:
    print('Finish STUDENT TODO 1 before calculating the baseline.')

## STUDENT TODO 2 — Build the preprocessing and regression pipeline

The pipeline should:

1. one-hot encode `day_of_week` and `month`,
2. pass the numerical features through unchanged,
3. fit a `LinearRegression` model.

Use `handle_unknown='ignore'` so a new category does not crash the application. Use `drop='first'` so one category from each encoded feature becomes the reference category.

In [ ]:
# STUDENT TODO 2: BUILD THE SCIKIT-LEARN PIPELINE
# Useful classes:
#   ColumnTransformer
#   OneHotEncoder
#   Pipeline
#   LinearRegression
#
# Suggested step names:
#   'preprocessing'
#   'regression'

preprocessor = None
model_pipeline = None

print('Build the preprocessing and regression pipeline.')

In [ ]:
def check_pipeline(model_pipeline):
    if model_pipeline is None:
        print('The model pipeline is not defined.')
        return False

    correct_type = isinstance(
        model_pipeline,
        Pipeline,
    )

    has_steps = (
        correct_type
        and 'preprocessing' in model_pipeline.named_steps
        and 'regression' in model_pipeline.named_steps
    )

    correct_preprocessor = (
        has_steps
        and isinstance(
            model_pipeline.named_steps['preprocessing'],
            ColumnTransformer,
        )
    )

    correct_regressor = (
        has_steps
        and isinstance(
            model_pipeline.named_steps['regression'],
            LinearRegression,
        )
    )

    passed = (
        correct_type
        and has_steps
        and correct_preprocessor
        and correct_regressor
    )

    print('Pipeline object:', correct_type)
    print('ColumnTransformer included:', correct_preprocessor)
    print('LinearRegression included:', correct_regressor)
    print('Passed!' if passed else 'Revise the pipeline.')

    if passed:
        print('\nPipeline:')
        print(model_pipeline)

    return passed


pipeline_ready = check_pipeline(
    model_pipeline
)

## STUDENT TODO 3 — Train and predict

Fit the pipeline on `X_train` and `y_train`. Then predict attendance for `X_test`.

Attendance cannot be negative, so clip predictions at zero.

In [ ]:
# STUDENT TODO 3: TRAIN THE MODEL AND PREDICT
# Required actions:
#   model_pipeline.fit(...)
#   model_pipeline.predict(...)
#   np.clip(..., 0, None)
#
# Save the final predictions as test_predictions.

test_predictions = None

print('Train the model and create test predictions.')

In [ ]:
def check_predictions(
    model_pipeline,
    test_predictions,
):
    if (
        model_pipeline is None
        or test_predictions is None
        or X_test is None
    ):
        print('Training and prediction are not complete.')
        return False

    predictions = np.asarray(
        test_predictions
    )

    correct_length = (
        len(predictions) == len(X_test)
    )

    finite_values = np.isfinite(
        predictions
    ).all()

    nonnegative_values = (
        predictions >= 0
    ).all()

    passed = (
        correct_length
        and finite_values
        and nonnegative_values
    )

    print('Prediction count:', len(predictions))
    print('Expected count:', len(X_test))
    print('All values finite:', finite_values)
    print('All values nonnegative:', nonnegative_values)
    print('Passed!' if passed else 'Check training, prediction, and clipping.')

    if passed:
        print(
            '\nFirst ten predictions:',
            np.round(predictions[:10], 1),
        )

    return passed


predictions_ready = check_predictions(
    model_pipeline,
    test_predictions,
)

## STUDENT TODO 4 — Evaluate the model

Calculate:

- `TEST_MAE`
- `TEST_RMSE`
- `TEST_R2`
- `IMPROVEMENT_OVER_BASELINE`

The percentage improvement is

$
rac{
	ext{baseline MAE}-	ext{regression MAE}
}{
	ext{baseline MAE}
}
	imes100.
$

In [ ]:
# STUDENT TODO 4: CALCULATE THE TEST METRICS
# Useful functions:
#   mean_absolute_error(...)
#   mean_squared_error(...)
#   np.sqrt(...)
#   r2_score(...)

TEST_MAE = None
TEST_RMSE = None
TEST_R2 = None
IMPROVEMENT_OVER_BASELINE = None

print('Calculate the regression evaluation metrics.')

In [ ]:
def check_evaluation_metrics(
    test_mae,
    test_rmse,
    test_r2,
    improvement_over_baseline,
):
    metric_values = [
        test_mae,
        test_rmse,
        test_r2,
        improvement_over_baseline,
    ]

    if (
        BASELINE_MAE is None
        or any(value is None for value in metric_values)
    ):
        print('The evaluation metrics are not complete.')
        return False

    finite_values = all(
        np.isfinite(value)
        for value in metric_values
    )

    sensible_errors = (
        test_mae >= 0
        and test_rmse >= 0
    )

    calculation_matches = np.isclose(
        improvement_over_baseline,
        (
            (BASELINE_MAE - test_mae)
            / BASELINE_MAE
            * 100
        ),
    )

    passed = (
        finite_values
        and sensible_errors
        and calculation_matches
    )

    evaluation_summary = pd.DataFrame(
        {
            'metric': [
                'Baseline MAE',
                'Regression MAE',
                'Regression RMSE',
                'Regression R-squared',
                'MAE improvement over baseline (%)',
            ],
            'value': [
                BASELINE_MAE,
                test_mae,
                test_rmse,
                test_r2,
                improvement_over_baseline,
            ],
        }
    )

    display(
        evaluation_summary.round(3)
    )

    if test_mae < BASELINE_MAE:
        print(
            'The regression model beats the baseline on MAE.'
        )
    else:
        print(
            'The regression model does not beat the baseline on MAE.'
        )

    print('Passed!' if passed else 'Check the metric formulas.')

    return passed


evaluation_ready = check_evaluation_metrics(
    TEST_MAE,
    TEST_RMSE,
    TEST_R2,
    IMPROVEMENT_OVER_BASELINE,
)

## Interpret the metrics

- **MAE** is the average absolute difference between predicted and actual attendance.
- **RMSE** penalizes especially large mistakes.
- **R-squared** measures how much variation in the later test dates is explained by the model.
- **Improvement over baseline** shows whether regression is more useful than always predicting the training average.

A strong average score does not mean every individual forecast is accurate.

In [ ]:
evaluation_data = None

if evaluation_ready:
    evaluation_data = test_data[
        [
            'date',
            'day_of_week',
            'is_holiday',
            'reserved_visitors',
            'reservation_count',
            'visitors',
        ]
    ].copy()

    evaluation_data = evaluation_data.rename(
        columns={
            'visitors': 'actual_visitors',
        }
    )

    evaluation_data['predicted_visitors'] = (
        test_predictions
    )

    evaluation_data['error'] = (
        evaluation_data['actual_visitors']
        - evaluation_data['predicted_visitors']
    )

    evaluation_data['absolute_error'] = (
        evaluation_data['error'].abs()
    )

    display(
        evaluation_data.head(10).round(2)
    )

else:
    print('Finish the four required TODOs before inspecting errors.')

In [ ]:
if evaluation_data is None:
    print('Finish the evaluation first.')

else:
    plt.figure(figsize=(14, 5))

    plt.plot(
        evaluation_data['date'],
        evaluation_data['actual_visitors'],
        marker='o',
        markersize=3,
        linewidth=1,
        label='Actual visitors',
    )

    plt.plot(
        evaluation_data['date'],
        evaluation_data['predicted_visitors'],
        linewidth=2,
        label='Predicted visitors',
    )

    plt.xlabel('Date')
    plt.ylabel('Visitors')
    plt.title('Actual and Predicted Attendance on Test Dates')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
if evaluation_data is None:
    print('Finish the evaluation first.')

else:
    largest_errors = (
        evaluation_data
        .sort_values(
            'absolute_error',
            ascending=False,
        )
        .head(10)
    )

    display(
        largest_errors.round(2)
    )

# Inspecting the Learned Weights

A positive weight increases the prediction relative to the model's reference categories. A negative weight decreases it.

Because the encoder uses `drop='first'`, one weekday and one month are omitted as reference categories.

Weights describe associations learned from this dataset. They do not prove that changing a feature would cause attendance to change.

In [ ]:
def extract_linear_model_weights(
    fitted_pipeline,
):
    """Extract the intercept and transformed linear-regression weights."""
    fitted_preprocessor = (
        fitted_pipeline
        .named_steps['preprocessing']
    )

    fitted_encoder = (
        fitted_preprocessor
        .named_transformers_['categorical']
    )

    encoded_feature_names = list(
        fitted_encoder.get_feature_names_out(
            CATEGORICAL_FEATURES
        )
    )

    transformed_feature_names = (
        encoded_feature_names
        + NUMERIC_FEATURES
    )

    regression_model = (
        fitted_pipeline
        .named_steps['regression']
    )

    weights = pd.DataFrame(
        {
            'feature': transformed_feature_names,
            'weight': regression_model.coef_,
        }
    )

    intercept = float(
        regression_model.intercept_
    )

    return intercept, weights

In [ ]:
EVALUATION_INTERCEPT = None
EVALUATION_WEIGHTS = None

if predictions_ready:
    (
        EVALUATION_INTERCEPT,
        EVALUATION_WEIGHTS,
    ) = extract_linear_model_weights(
        model_pipeline
    )

    weights_for_display = (
        EVALUATION_WEIGHTS
        .assign(
            absolute_weight=lambda frame:
            frame['weight'].abs()
        )
        .sort_values(
            'absolute_weight',
            ascending=False,
        )
    )

    print(
        f'Model intercept: '
        f'{EVALUATION_INTERCEPT:.3f}'
    )

    display(
        weights_for_display.round(3)
    )

else:
    print('Train the model before inspecting its weights.')

# Train the Final Forecasting Model

The chronological split was necessary for honest evaluation.

After evaluation is complete, fit a fresh copy of the same pipeline on all available historical rows. This final model will power the future-day application.

In [ ]:
final_model = None
FINAL_INTERCEPT = None
FINAL_WEIGHTS = None

if evaluation_ready:
    final_model = clone(
        model_pipeline
    )

    final_model.fit(
        data[FEATURES],
        data[TARGET],
    )

    (
        FINAL_INTERCEPT,
        FINAL_WEIGHTS,
    ) = extract_linear_model_weights(
        final_model
    )

    print(
        f'Final model trained on '
        f'{len(data):,} operating days.'
    )

    print(
        f'Final model intercept: '
        f'{FINAL_INTERCEPT:.3f}'
    )

else:
    print('Finish evaluation before training the final model.')

In [ ]:
if FINAL_WEIGHTS is None:
    print('The final model has not been trained.')

else:
    final_weights_for_display = (
        FINAL_WEIGHTS
        .assign(
            absolute_weight=lambda frame:
            frame['weight'].abs()
        )
        .sort_values(
            'absolute_weight',
            ascending=False,
        )
    )

    display(
        final_weights_for_display.round(3)
    )

# Build the Forecasting Application

The trained regression model can now estimate attendance for a future operating-day scenario.

The application will:

1. validate the five user-provided values,
2. create a one-row pandas DataFrame,
3. apply the same preprocessing pipeline used during training,
4. calculate the attendance prediction,
5. display a rough planning range based on test MAE,
6. warn when reservation inputs exceed the historical range.

## Why are weather and campus events absent?

A model can learn from a feature only when historical values for that feature are available during training.

This dataset does not contain historical weather or campus-event information. Adding those fields only at prediction time would not teach the model how they relate to attendance.

## About the planning range

The range shown below is the prediction plus or minus the model's test MAE. It is an easy-to-explain operational estimate, not a formal statistical confidence interval.

In [ ]:
VALID_WEEKDAYS = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday',
]


def validate_forecast_inputs(
    day_of_week,
    month,
    is_holiday,
    reserved_visitors,
    reservation_count,
):
    """Check that the future-day inputs are internally sensible."""
    if day_of_week not in VALID_WEEKDAYS:
        raise ValueError(
            'day_of_week must be one of: '
            + ', '.join(VALID_WEEKDAYS)
        )

    if not 1 <= month <= 12:
        raise ValueError(
            'month must be between 1 and 12.'
        )

    if is_holiday not in [0, 1]:
        raise ValueError(
            'is_holiday must be 0 or 1.'
        )

    if reserved_visitors < 0:
        raise ValueError(
            'reserved_visitors cannot be negative.'
        )

    if reservation_count < 0:
        raise ValueError(
            'reservation_count cannot be negative.'
        )

    if (
        reserved_visitors == 0
        and reservation_count > 0
    ):
        raise ValueError(
            'There cannot be reservations with '
            'zero total reserved visitors.'
        )

    if (
        reserved_visitors > 0
        and reservation_count == 0
    ):
        raise ValueError(
            'A positive number of reserved visitors '
            'requires at least one reservation.'
        )

    if reservation_count > reserved_visitors:
        raise ValueError(
            'The number of reservations cannot exceed '
            'the total number of reserved visitors.'
        )

In [ ]:
def find_input_warnings(
    reserved_visitors,
    reservation_count,
):
    """Warn when an input is outside the historical training range."""
    warnings = []

    maximum_historical_reserved_visitors = (
        data['reserved_visitors'].max()
    )

    maximum_historical_reservation_count = (
        data['reservation_count'].max()
    )

    if (
        reserved_visitors
        > maximum_historical_reserved_visitors
    ):
        warnings.append(
            'Reserved visitors is larger than any '
            'value in the historical dataset.'
        )

    if (
        reservation_count
        > maximum_historical_reservation_count
    ):
        warnings.append(
            'Reservation count is larger than any '
            'value in the historical dataset.'
        )

    return warnings

In [ ]:
def forecast_cafeteria_crowd(
    day_of_week,
    month,
    is_holiday,
    reserved_visitors,
    reservation_count,
):
    """Predict attendance for one future operating-day scenario."""
    if final_model is None:
        raise RuntimeError(
            'Finish the four required TODOs and train the final model first.'
        )

    day_of_week = (
        str(day_of_week)
        .strip()
        .title()
    )

    month = int(month)
    is_holiday = int(is_holiday)
    reserved_visitors = int(
        reserved_visitors
    )
    reservation_count = int(
        reservation_count
    )

    validate_forecast_inputs(
        day_of_week=day_of_week,
        month=month,
        is_holiday=is_holiday,
        reserved_visitors=reserved_visitors,
        reservation_count=reservation_count,
    )

    input_values = {
        'day_of_week': day_of_week,
        'month': month,
        'is_holiday': is_holiday,
        'reserved_visitors': reserved_visitors,
        'reservation_count': reservation_count,
    }

    new_operating_day = pd.DataFrame(
        [input_values]
    )

    prediction = float(
        final_model.predict(
            new_operating_day[FEATURES]
        )[0]
    )

    prediction = max(
        0,
        prediction,
    )

    # Easy-to-explain operational range.
    # This is not a formal confidence interval.
    planning_low = max(
        0,
        prediction - TEST_MAE,
    )

    planning_high = (
        prediction + TEST_MAE
    )

    input_warnings = find_input_warnings(
        reserved_visitors=reserved_visitors,
        reservation_count=reservation_count,
    )

    print('=' * 60)
    print('CAFETERIA CROWD FORECAST')
    print('=' * 60)

    print(f'Day: {day_of_week}')
    print(f'Month: {month}')

    print(
        'Holiday:',
        'Yes' if is_holiday == 1 else 'No',
    )

    print(
        'Advance reserved visitors:',
        reserved_visitors,
    )

    print(
        'Advance reservation count:',
        reservation_count,
    )

    print('-' * 60)

    print(
        'Predicted attendance:',
        f'{prediction:.1f} visitors',
    )

    print(
        'Rough planning range:',
        f'{planning_low:.1f} to '
        f'{planning_high:.1f} visitors',
    )

    print(
        'Typical test error:',
        f'{TEST_MAE:.1f} visitors',
    )

    print(
        'Test R-squared:',
        f'{TEST_R2:.3f}',
    )

    if input_warnings:
        print('\nWarnings:')

        for warning in input_warnings:
            print('-', warning)

    return {
        'inputs': input_values,
        'prediction': prediction,
        'planning_low': planning_low,
        'planning_high': planning_high,
        'test_mae': TEST_MAE,
        'test_rmse': TEST_RMSE,
        'test_r2': TEST_R2,
        'warnings': input_warnings,
    }

In [ ]:
def main(
    day_of_week,
    month,
    is_holiday,
    reserved_visitors,
    reservation_count,
):
    """Run the complete crowd-forecasting application."""
    try:
        return forecast_cafeteria_crowd(
            day_of_week=day_of_week,
            month=month,
            is_holiday=is_holiday,
            reserved_visitors=reserved_visitors,
            reservation_count=reservation_count,
        )

    except (ValueError, RuntimeError) as error:
        print('The forecast could not be completed:')
        print(error)
        return None

# Try a Future Operating Day

Change the five scenario values below.

Use:

- a complete weekday name,
- a month from 1 through 12,
- 1 for a holiday or 0 otherwise,
- nonnegative reservation values.

In [ ]:
# STUDENT CUSTOMIZATION:
# Enter a future operating-day scenario.

forecast_result = main(
    day_of_week='Friday',
    month=9,
    is_holiday=0,
    reserved_visitors=30,
    reservation_count=5,
)

## Reflection

1. Which exploratory pattern seemed most useful for prediction?
2. Why was a chronological split more appropriate than a random split?
3. What does one-hot encoding do?
4. Did regression beat the baseline? By how much?
5. Explain MAE in the context of this application.
6. Which test dates produced the largest errors?
7. Which learned weights had the largest magnitudes?
8. Why should model weights not be interpreted automatically as causal effects?
9. Why is the planning range not a formal confidence interval?
10. Why is the regression prediction kept separate from the LLM explanation?
11. What could happen when a future input is outside the historical range?
12. Name one useful feature that would require additional historical data.

## Optional controlled experiment

Change exactly one design choice and compare the results:

- use `Ridge` regression instead of ordinary linear regression,
- add or remove one defensible feature,
- change the training fraction,
- use a different baseline,
- inspect performance separately by weekday,
- change the future scenario,
- revise the communication prompt without changing the numerical forecast.

Record the baseline MAE, regression MAE, RMSE, R-squared, and your interpretation.